# ClickTheLook — Standalone Training Pipeline

Run all cells top-to-bottom. **Only the Configuration cell needs editing between runs.**


In [ ]:
import os

# =============================================================================
# CONFIGURATION — Edit this cell only
# =============================================================================

# -- Paths --------------------------------------------------------------------
DATA_ROOT           = "/path/to/your/data"   # root with deepfashion2_original_images/ & img_info_dataframes/
YOLO_DATASET_DIR    = "./data/yolo"           # YOLO images/ and labels/ will be created here
TRAINING_OUTPUT_DIR = "./artifacts/runs"
WEIGHTS_DIR         = "./artifacts/weights"
EXPORTS_DIR         = "./artifacts/exports"

# -- MLflow -------------------------------------------------------------------
MLFLOW_TRACKING_URI = "./mlruns"             # local folder; change to shared path on HPC
MLFLOW_EXPERIMENT   = "ClickTheLook"

# -- Model --------------------------------------------------------------------
MODEL    = "yolo11n.pt"
RUN_NAME = "yolo11_deepfashion2"

# -- Core training ------------------------------------------------------------
EPOCHS    = 1
BATCH     = 64
IMGSZ     = 512
PATIENCE  = 10
OPTIMIZER = "AdamW"

# -- Learning rate schedule ---------------------------------------------------
LR0             = 0.01
LRF             = 0.1
MOMENTUM        = 0.937
WEIGHT_DECAY    = 0.0005
WARMUP_EPOCHS   = 1.0
WARMUP_MOMENTUM = 0.8
WARMUP_BIAS_LR  = 0.1
COS_LR          = True

# -- Regularisation -----------------------------------------------------------
LABEL_SMOOTHING = 0.1

# -- Augmentation -------------------------------------------------------------
HSV_H       = 0.015
HSV_S       = 0.2
HSV_V       = 0.2
DEGREES     = 5.0
TRANSLATE   = 0.05
SCALE       = 0.2
SHEAR       = 0.0
PERSPECTIVE = 0.0
FLIPUD      = 0.0
FLIPLR      = 0.5
MOSAIC      = 0.5
MIXUP       = 0.0

# -- Hardware & runtime -------------------------------------------------------
WORKERS = 8
AMP     = True
CACHE   = False   # False | ram | disk
VAL     = False   # True to validate after every epoch

print("Configuration set.")


In [ ]:
import gc
import json
import random
import shutil
import time
from ast import literal_eval
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} ({props.total_memory / 1024**3:.1f} GB)")


In [ ]:
# ── Dataset classes ───────────────────────────────────────────────────────────
CATEGORIES = {
    1: "short_sleeve_top",    2: "long_sleeve_top",
    3: "short_sleeve_outwear",4: "long_sleeve_outwear",
    5: "vest",                6: "sling",
    7: "shorts",              8: "trousers",
    9: "skirt",              10: "short_sleeve_dress",
   11: "long_sleeve_dress",  12: "vest_dress",
   13: "sling_dress",
}
NUM_CLASSES = len(CATEGORIES)
class_names = [CATEGORIES[i + 1] for i in range(NUM_CLASSES)]

# ── Local dataset paths ───────────────────────────────────────────────────────
LOCAL_PATHS = {
    "train_csv":    os.path.join(DATA_ROOT, "img_info_dataframes", "train.csv"),
    "val_csv":      os.path.join(DATA_ROOT, "img_info_dataframes", "validation.csv"),
    "train_images": os.path.join(DATA_ROOT, "deepfashion2_original_images", "train", "image"),
    "val_images":   os.path.join(DATA_ROOT, "deepfashion2_original_images", "validation", "image"),
}

# ── Data loading parameters ───────────────────────────────────────────────────
CSV_BATCH_SIZE       = 20000
IMAGE_DOWNLOAD_BATCH = 200
MIN_DISK_FREE_GB     = 5.0

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# ── Training config ───────────────────────────────────────────────────────────
TRAINING_CONFIG = {
    "model":            MODEL,
    "epochs":           EPOCHS,
    "batch":            BATCH,
    "imgsz":            IMGSZ,
    "patience":         PATIENCE,
    "optimizer":        OPTIMIZER,
    "lr0":              LR0,
    "lrf":              LRF,
    "momentum":         MOMENTUM,
    "weight_decay":     WEIGHT_DECAY,
    "warmup_epochs":    WARMUP_EPOCHS,
    "warmup_momentum":  WARMUP_MOMENTUM,
    "warmup_bias_lr":   WARMUP_BIAS_LR,
    "cos_lr":           COS_LR,
    "label_smoothing":  LABEL_SMOOTHING,
    "hsv_h":            HSV_H,
    "hsv_s":            HSV_S,
    "hsv_v":            HSV_V,
    "degrees":          DEGREES,
    "translate":        TRANSLATE,
    "scale":            SCALE,
    "shear":            SHEAR,
    "perspective":      PERSPECTIVE,
    "flipud":           FLIPUD,
    "fliplr":           FLIPLR,
    "mosaic":           MOSAIC,
    "mixup":            MIXUP,
    "amp":              AMP,
    "device":           DEVICE,
    "workers":          WORKERS,
    "cache":            CACHE,
    "val":              VAL,
    "project":          TRAINING_OUTPUT_DIR,
    "name":             RUN_NAME,
    "exist_ok":         True,
    "pretrained":       True,
    "verbose":          True,
}

if torch.cuda.device_count() > 1:
    TRAINING_CONFIG["device"] = list(range(torch.cuda.device_count()))
    TRAINING_CONFIG["batch"] *= torch.cuda.device_count()

for _d in [YOLO_DATASET_DIR, TRAINING_OUTPUT_DIR, WEIGHTS_DIR, EXPORTS_DIR]:
    os.makedirs(_d, exist_ok=True)

print(f"Device     : {TRAINING_CONFIG['device']}")
print(f"Model      : {TRAINING_CONFIG['model']}")
print(f"Epochs     : {TRAINING_CONFIG['epochs']} | Batch: {TRAINING_CONFIG['batch']} | imgsz: {TRAINING_CONFIG['imgsz']}")
print(f"Data root  : {DATA_ROOT}")
print(f"YOLO dir   : {YOLO_DATASET_DIR}")
print(f"Weights dir: {WEIGHTS_DIR}")


## Function Definitions

### Utilities

In [ ]:
def get_disk_free_gb(path: str) -> float:
    stat = os.statvfs(path)
    return (stat.f_bavail * stat.f_frsize) / (1024 ** 3)


def verify_local_paths():
    print("Verifying data paths...")
    print("=" * 60)
    ok = True
    for name, path in LOCAL_PATHS.items():
        exists = os.path.exists(path)
        print(f"  {'OK' if exists else 'MISSING'} {name}: {path}")
        if not exists:
            ok = False
    print("=" * 60)
    return ok

print("Utility functions defined.")


### Data Loader

In [ ]:
class LocalDataLoader:
    def __init__(self, csv_batch_size=20000, image_batch=200, min_disk_free_gb=5.0):
        self.csv_batch_size   = csv_batch_size
        self.image_batch      = image_batch
        self.min_disk_free_gb = min_disk_free_gb

    def stream_csv_batches(self, csv_path: str):
        print(f"  Reading CSV: {csv_path}")
        size_mb    = os.path.getsize(csv_path) / 1024**2
        total_rows = sum(1 for _ in open(csv_path)) - 1
        total_batches = (total_rows // self.csv_batch_size) + 1
        print(f"  CSV size: {size_mb:.1f} MB | ~{total_rows:,} rows | {total_batches} batches")
        batch_num = 0
        for chunk in tqdm(pd.read_csv(csv_path, chunksize=self.csv_batch_size),
                          total=total_batches, desc="  CSV batches", unit="batch"):
            batch_num += 1
            yield batch_num, chunk
        print(f"  Streamed {batch_num} CSV batches")

    def load_csv(self, csv_path: str) -> pd.DataFrame:
        print(f"  Loading: {csv_path}")
        df = pd.read_csv(csv_path)
        print(f"  Loaded {len(df):,} rows ({df.memory_usage(deep=True).sum()/1024**2:.1f} MB)")
        return df

    def build_image_index(self, images_dir: str) -> Dict[str, str]:
        images_dir = os.path.abspath(images_dir)
        print(f"  Indexing: {images_dir}")
        index = {
            fname: os.path.join(images_dir, fname)
            for fname in os.listdir(images_dir)
            if fname.lower().endswith((".jpg", ".jpeg", ".png"))
        }
        print(f"  Found {len(index):,} images")
        return index

    def link_images(self, filenames: List[str], src_index: Dict[str, str],
                    dest_dir: str) -> Tuple[int, bool]:
        os.makedirs(dest_dir, exist_ok=True)
        linked  = 0
        to_link = [fn for fn in filenames
                   if fn in src_index and not os.path.exists(os.path.join(dest_dir, fn))]
        for i in range(0, len(to_link), self.image_batch):
            free_gb = get_disk_free_gb(YOLO_DATASET_DIR)
            if free_gb < self.min_disk_free_gb:
                print(f"\n  Disk low ({free_gb:.1f} GB). Stopping.")
                return linked, False
            for fn in to_link[i : i + self.image_batch]:
                dest = os.path.join(dest_dir, fn)
                try:
                    os.symlink(src_index[fn], dest)
                    linked += 1
                except OSError as e:
                    print(f"  Warning: {fn}: {e}")
        return linked, True


loader = LocalDataLoader(
    csv_batch_size=CSV_BATCH_SIZE,
    image_batch=IMAGE_DOWNLOAD_BATCH,
    min_disk_free_gb=MIN_DISK_FREE_GB,
)
print("LocalDataLoader defined + loader instance created.")


### Data Analysis

In [ ]:
def analyze_metadata(df, split_name):
    print(f"\n{'='*60}\n{split_name.upper()} SPLIT\n{'='*60}")
    print(f"Total annotations: {len(df):,}")
    if "path" in df.columns:
        u = df["path"].nunique()
        print(f"Unique images: {u:,} | Avg annot/img: {len(df)/u:.1f}")
    cat_counts = df["category_id"].value_counts().sort_index()
    for cid, cnt in cat_counts.items():
        print(f"  {cid:2d}. {CATEGORIES.get(cid,'?'):25s}: {cnt:>7,} ({cnt/len(df)*100:5.1f}%)")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh([CATEGORIES.get(i, "?") for i in cat_counts.index],
                 cat_counts.values, color="steelblue")
    axes[0].set_title(f"{split_name} — Categories")
    if "scale" in df.columns:
        sc = df["scale"].value_counts().sort_index()
        axes[1].bar([{1:"Small",2:"Med",3:"Large"}.get(s,str(s)) for s in sc.index],
                    sc.values, color="coral")
        axes[1].set_title(f"{split_name} — Scale")
    plt.tight_layout()
    plt.show()

print("Data analysis functions defined.")


### Data Conversion

In [ ]:
def convert_bbox_to_yolo(bbox_str, img_w, img_h):
    try:
        bbox = literal_eval(bbox_str) if isinstance(bbox_str, str) else bbox_str
        x1, y1, x2, y2 = bbox
        x1, x2 = max(0, min(x1, img_w)), max(0, min(x2, img_w))
        y1, y2 = max(0, min(y1, img_h)), max(0, min(y2, img_h))
        w, h = (x2 - x1) / img_w, (y2 - y1) / img_h
        if w <= 0 or h <= 0:
            return None
        return [((x1 + x2) / 2) / img_w, ((y1 + y2) / 2) / img_h, w, h]
    except:
        return None


def convert_split(split, csv_path, images_dir):
    img_dir = os.path.join(YOLO_DATASET_DIR, "images", split)
    lbl_dir = os.path.join(YOLO_DATASET_DIR, "labels", split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    stats = dict(total=0, converted=0, skipped=0, linked=0,
                 cached=0, labels=0, batches=0, disk_stop=False)
    src_index = loader.build_image_index(images_dir)

    print(f"\nConverting {split} in batches...")
    print("=" * 60)

    for batch_num, chunk_df in loader.stream_csv_batches(csv_path):
        stats["batches"] = batch_num
        t0 = time.time()
        fnames = [os.path.basename(p) for p in chunk_df["path"].unique()]
        cached = sum(1 for fn in fnames if os.path.exists(os.path.join(img_dir, fn)))
        stats["cached"] += cached
        n_linked, disk_ok = loader.link_images(fnames, src_index, img_dir)
        stats["linked"] += n_linked
        if not disk_ok:
            stats["disk_stop"] = True
            print(f"  Batch {batch_num}: DISK FULL — stopping.")
            del chunk_df; gc.collect(); break

        for path, group in chunk_df.groupby("path"):
            fn = os.path.basename(path)
            if not os.path.exists(os.path.join(img_dir, fn)):
                stats["skipped"] += len(group); continue
            lines = []
            for _, row in group.iterrows():
                stats["total"] += 1
                cid = int(row["category_id"]) - 1
                if cid < 0 or cid >= NUM_CLASSES:
                    stats["skipped"] += 1; continue
                yolo = convert_bbox_to_yolo(
                    row["b_box"], int(row.get("img_width", 640)), int(row.get("img_height", 640)))
                if yolo is None:
                    stats["skipped"] += 1; continue
                lines.append(f"{cid} {yolo[0]:.6f} {yolo[1]:.6f} {yolo[2]:.6f} {yolo[3]:.6f}")
                stats["converted"] += 1
            if lines:
                lbl_path = os.path.join(lbl_dir, os.path.splitext(fn)[0] + ".txt")
                mode = "a" if os.path.exists(lbl_path) else "w"
                with open(lbl_path, mode) as f:
                    f.write("\n".join(lines) + "\n")
                stats["labels"] += 1

        free = get_disk_free_gb(YOLO_DATASET_DIR)
        print(f"  Batch {batch_num}: {len(chunk_df):,} annot | +{n_linked} linked | "
              f"{time.time()-t0:.1f}s | {free:.1f} GB free")
        del chunk_df; gc.collect()

    print(f"\n{'='*60}")
    print(f"{split.upper()} DONE — {stats['converted']:,}/{stats['total']:,} annotations, "
          f"{stats['linked']:,} linked, {stats['labels']:,} label files")
    if stats["disk_stop"]:
        print("  Stopped early due to disk space.")
    print("=" * 60)
    return stats

print("Data conversion functions defined.")


### Dataset

In [ ]:
def labels_exist() -> bool:
    for split in ["train", "val"]:
        ldir = os.path.join(YOLO_DATASET_DIR, "labels", split)
        if not os.path.exists(ldir) or not any(f.endswith(".txt") for f in os.listdir(ldir)):
            return False
    return True


def symlinks_valid() -> bool:
    for split in ["train", "val"]:
        idir = os.path.join(YOLO_DATASET_DIR, "images", split)
        if not os.path.exists(idir):
            return False
        if not any(os.path.exists(os.path.join(idir, f))
                   for f in os.listdir(idir) if f.endswith((".jpg", ".jpeg", ".png"))):
            return False
    return True


def restore_symlinks():
    for split in ["train", "val"]:
        idir = os.path.join(YOLO_DATASET_DIR, "images", split)
        ldir = os.path.join(YOLO_DATASET_DIR, "labels", split)
        os.makedirs(idir, exist_ok=True)
        src_index   = loader.build_image_index(LOCAL_PATHS[f"{split}_images"])
        label_stems = {os.path.splitext(f)[0] for f in os.listdir(ldir) if f.endswith(".txt")}
        fnames      = [fn for fn in src_index if os.path.splitext(fn)[0] in label_stems]
        linked, _   = loader.link_images(fnames, src_index, idir)
        print(f"  {split}: {linked} symlinks restored.")


def verify_dataset():
    print("\nYOLO dataset:")
    for split in ["train", "val"]:
        idir = os.path.join(YOLO_DATASET_DIR, "images", split)
        ldir = os.path.join(YOLO_DATASET_DIR, "labels", split)
        ni = len([f for f in os.listdir(idir) if f.endswith((".jpg",".jpeg",".png"))]) if os.path.exists(idir) else 0
        nl = len([f for f in os.listdir(ldir) if f.endswith(".txt")]) if os.path.exists(ldir) else 0
        print(f"  {split}: {ni:,} images, {nl:,} labels")
    print(f"\nDisk free: {get_disk_free_gb(YOLO_DATASET_DIR):.1f} GB")


def create_dataset_yaml() -> str:
    cfg = {"path": os.path.abspath(YOLO_DATASET_DIR), "train": "images/train",
           "val": "images/val", "nc": NUM_CLASSES, "names": class_names}
    yaml_path = os.path.join(YOLO_DATASET_DIR, "dataset.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"YAML: {yaml_path}")
    with open(yaml_path) as f:
        print(f.read())
    return yaml_path

print("Dataset functions defined.")


### Training

In [ ]:
def load_model() -> YOLO:
    best_path = os.path.join(WEIGHTS_DIR, "best.pt")
    if os.path.exists(best_path):
        print(f"Fine-tuning from existing best: {best_path}")
        return YOLO(best_path)
    print(f"No checkpoint found — training from scratch: {TRAINING_CONFIG['model']}")
    return YOLO(TRAINING_CONFIG["model"])


def train_model(model: YOLO, yaml_path: str):
    print("=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    print(f"Dataset : {yaml_path}")
    print(f"Device  : {TRAINING_CONFIG['device']}")
    print(f"Batch   : {TRAINING_CONFIG['batch']}")
    print()
    params = {k: v for k, v in TRAINING_CONFIG.items() if k != "model"}
    results = model.train(data=yaml_path, **params)
    print("\nTraining complete!")
    return results


def export_model(model: YOLO):
    export_path = model.export(format="onnx", dynamic=True)
    dest = os.path.join(EXPORTS_DIR, os.path.basename(export_path))
    shutil.move(str(export_path), dest)
    print(f"ONNX export: {dest}")


def _load_scores(scores_path: str) -> dict:
    if os.path.exists(scores_path):
        with open(scores_path) as f:
            return json.load(f)
    return {}


def _save_scores(scores_path: str, scores: dict):
    with open(scores_path, "w") as f:
        json.dump(scores, f, indent=2)


def update_model_registry(model: YOLO, metrics):
    # Returns (should_export: bool, status: str)
    # status is one of: "best" | "last" | "discarded"
    scores_path = os.path.join(WEIGHTS_DIR, "scores.json")
    new_map50   = float(metrics.box.map50)
    scores      = _load_scores(scores_path)
    best_map50  = scores.get("best")
    last_map50  = scores.get("last")

    run_best = os.path.join(TRAINING_OUTPUT_DIR,
                            TRAINING_CONFIG.get("name", "yolo11_deepfashion2"),
                            "weights", "best.pt")

    print("\n" + "=" * 60)
    print("MODEL REGISTRY COMPARISON")
    print("=" * 60)
    print(f"  New   mAP50 : {new_map50:.4f}")
    print(f"  Best  mAP50 : {best_map50:.4f}" if best_map50 is not None
          else "  Best  mAP50 : (none — first run)")
    print(f"  Last  mAP50 : {last_map50:.4f}" if last_map50 is not None
          else "  Last  mAP50 : (none)")

    if not os.path.exists(run_best):
        print(f"  WARNING: run checkpoint not found at {run_best}. Skipping update.")
        return False, "discarded"

    best_dest = os.path.join(WEIGHTS_DIR, "best.pt")
    last_dest = os.path.join(WEIGHTS_DIR, "last.pt")

    if best_map50 is None or new_map50 > best_map50:
        if os.path.exists(best_dest):
            shutil.copy2(best_dest, last_dest)
            scores["last"] = best_map50
        shutil.copy2(run_best, best_dest)
        scores["best"] = new_map50
        _save_scores(scores_path, scores)
        print(f"  -> NEW BEST  ({new_map50:.4f}). Exporting.")
        return True, "best"

    if last_map50 is None or new_map50 > last_map50:
        shutil.copy2(run_best, last_dest)
        scores["last"] = new_map50
        _save_scores(scores_path, scores)
        print(f"  -> New LAST  ({new_map50:.4f}). Saved, not exported.")
        return False, "last"

    print(f"  -> DISCARDED ({new_map50:.4f} <= last {last_map50:.4f}). Nothing saved.")
    return False, "discarded"

print("Training functions defined.")


### Evaluation

In [ ]:
def load_best_model(current_model: YOLO) -> YOLO:
    best_path = os.path.join(TRAINING_OUTPUT_DIR,
                             TRAINING_CONFIG.get("name", "yolo11_deepfashion2"),
                             "weights", "best.pt")
    if os.path.exists(best_path):
        print(f"Loading best checkpoint from current run: {best_path}")
        return YOLO(best_path)
    print("Best checkpoint not found — using model from memory.")
    return current_model


def run_validation(model: YOLO, yaml_path: str):
    metrics = model.val(data=yaml_path)
    print(f"\n{'='*60}\nVALIDATION RESULTS\n{'='*60}")
    print(f"mAP50    : {metrics.box.map50:.4f}")
    print(f"mAP50-95 : {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall   : {metrics.box.mr:.4f}")
    print("\nPer-class AP50:")
    for name, ap in zip(class_names, metrics.box.ap50):
        print(f"  {name:25s}: {ap:.4f}")
    return metrics


def visualize_results():
    rdir = os.path.join(TRAINING_OUTPUT_DIR, TRAINING_CONFIG.get("name", "yolo11_deepfashion2"))
    for fname, title in [("results.png", "Training Results"),
                         ("confusion_matrix.png", "Confusion Matrix")]:
        p = os.path.join(rdir, fname)
        if os.path.exists(p):
            plt.figure(figsize=(15, 10) if "results" in fname else (12, 10))
            plt.imshow(Image.open(p))
            plt.axis("off")
            plt.title(title)
            plt.show()

print("Evaluation functions defined.")


### Inference

In [ ]:
def run_inference(model: YOLO, image_path: str, conf: float = 0.5) -> dict:
    results = model(image_path, conf=conf)
    dets = []
    for r in results:
        for box in r.boxes:
            dets.append({
                "class_name": CATEGORIES[int(box.cls[0]) + 1],
                "confidence": float(box.conf[0]),
                "bbox":       box.xyxy[0].tolist(),
            })
    return {"image_path": image_path, "detections": dets, "results": results}


def visualize_inference(result: dict):
    ann = result["results"][0].plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"Detected {len(result['detections'])} items")
    plt.show()
    for d in result["detections"]:
        print(f"  {d['class_name']}: {d['confidence']:.2f}")


def run_sample_inference(model: YOLO):
    vdir = os.path.join(YOLO_DATASET_DIR, "images", "val")
    if os.path.exists(vdir):
        imgs = [f for f in os.listdir(vdir) if f.endswith((".jpg", ".jpeg", ".png"))]
        for p in random.sample([os.path.join(vdir, f) for f in imgs], min(4, len(imgs))):
            print(f"\n{'='*60}\n{os.path.basename(p)}")
            visualize_inference(run_inference(model, p))

print("Inference functions defined.")


### Cleanup

In [ ]:
def cleanup(keep_weights: bool = True):
    if os.path.exists(YOLO_DATASET_DIR):
        shutil.rmtree(YOLO_DATASET_DIR)
        print(f"  Removed {YOLO_DATASET_DIR}")
    if not keep_weights and os.path.exists(TRAINING_OUTPUT_DIR):
        shutil.rmtree(TRAINING_OUTPUT_DIR)
        print(f"  Removed {TRAINING_OUTPUT_DIR}")
    gc.collect()
    print(f"  Disk free: {get_disk_free_gb(YOLO_DATASET_DIR):.1f} GB")

print("Cleanup function defined.")


## 1. Path Verification

In [ ]:
verify_local_paths()

## 2. Disk Space Check

In [ ]:
free = get_disk_free_gb(YOLO_DATASET_DIR)
print(f"Disk free: {free:.1f} GB")
if free < MIN_DISK_FREE_GB:
    print(f"  Warning: below minimum ({MIN_DISK_FREE_GB} GB)!")
elif free < 50:
    print("  Pipeline will stop gracefully if disk fills up.")
else:
    print("  Plenty of space.")


## 3. Data Exploration

In [ ]:
print("Loading training metadata...")
train_df = loader.load_csv(LOCAL_PATHS["train_csv"])
print(f"Shape: {train_df.shape}")
analyze_metadata(train_df, "train")

print("Loading validation metadata...")
val_df = loader.load_csv(LOCAL_PATHS["val_csv"])
analyze_metadata(val_df, "validation")

del train_df, val_df
gc.collect()
print("Freed exploration DataFrames.")


## 4. Data Conversion: DeepFashion2 → YOLO

In [ ]:
if not labels_exist():
    print("PHASE: Converting training data to YOLO format")
    convert_split("train", LOCAL_PATHS["train_csv"], LOCAL_PATHS["train_images"])
    print("\nPHASE: Converting validation data to YOLO format")
    convert_split("val", LOCAL_PATHS["val_csv"], LOCAL_PATHS["val_images"])
elif not symlinks_valid():
    print("Labels found. Symlinks broken — restoring without re-conversion...")
    restore_symlinks()
    print("Symlinks restored.")
else:
    print("YOLO-format data ready. Skipping conversion.")


## 5. Dataset Verification + YAML

In [ ]:
verify_dataset()
yaml_path = create_dataset_yaml()


## 6. Training

In [ ]:
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

# ── MLflow: initialise experiment and start run ───────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
active_run = mlflow.start_run(run_name=RUN_NAME)
print(f"MLflow run ID : {active_run.info.run_id}")
print(f"Experiment    : {MLFLOW_EXPERIMENT}")
print(f"Tracking URI  : {MLFLOW_TRACKING_URI}")

# Log every hyperparameter from TRAINING_CONFIG
mlflow.log_params({k: str(v) for k, v in TRAINING_CONFIG.items()})

# ── Train ─────────────────────────────────────────────────────────────────────
model  = load_model()
results = train_model(model, yaml_path)


## 7. Evaluation

In [ ]:
model   = load_best_model(model)
metrics = run_validation(model, yaml_path)
visualize_results()

# ── MLflow: log validation metrics ────────────────────────────────────────────
mlflow.log_metrics({
    "mAP50":     metrics.box.map50,
    "mAP50_95":  metrics.box.map,
    "precision": metrics.box.mp,
    "recall":    metrics.box.mr,
})
for cls_name, ap in zip(class_names, metrics.box.ap50):
    mlflow.log_metric(f"AP50_{cls_name}", ap)
print("Metrics logged to MLflow.")


## 8. Model Registry Comparison

Compares the new model against the globally managed `best.pt` and `last.pt`:
- **new > best** → new becomes best, old best becomes last → exports
- **last < new ≤ best** → new becomes last → no export
- **new ≤ last** → discarded → no export


In [ ]:
should_export, registry_status = update_model_registry(model, metrics)

# ── MLflow: tag the run outcome ────────────────────────────────────────────────
mlflow.set_tag("registry_result", registry_status)
mlflow.set_tag("base_model",      TRAINING_CONFIG["model"])
mlflow.set_tag("run_name",        RUN_NAME)
print(f"Registry result: {registry_status}")


## 9. Inference

In [ ]:
run_sample_inference(model)

## 10. Export *(only if new model earned the best slot)*

In [ ]:
if should_export:
    export_model(model)
else:
    print("Model did not improve over stored best. Skipping export.")

# ── MLflow: log artifacts and close run ───────────────────────────────────────
run_dir = os.path.join(TRAINING_OUTPUT_DIR, RUN_NAME)
for fname in ["results.png", "results.csv", "confusion_matrix.png"]:
    p = os.path.join(run_dir, fname)
    if os.path.exists(p):
        mlflow.log_artifact(p, artifact_path="training_artifacts")

best_pt = os.path.join(WEIGHTS_DIR, "best.pt")
if registry_status == "best" and os.path.exists(best_pt):
    mlflow.log_artifact(best_pt, artifact_path="weights")

scores_json = os.path.join(WEIGHTS_DIR, "scores.json")
if os.path.exists(scores_json):
    mlflow.log_artifact(scores_json, artifact_path="registry")

mlflow.end_run()
print(f"\nMLflow run closed.")
print(f"Dashboard : mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print(f"Then open : http://localhost:5000")


## 11. Cleanup *(disabled — uncomment to run)*

In [ ]:
# cleanup(keep_weights=True)


## Summary

In [ ]:
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"\nData root  : {DATA_ROOT}")
print(f"YOLO dir   : {YOLO_DATASET_DIR}")
print(f"Weights    : {WEIGHTS_DIR}/best.pt  (best), last.pt (second-best)")
print(f"Scores     : {WEIGHTS_DIR}/scores.json")
print(f"\nModel      : {TRAINING_CONFIG['model']} | {TRAINING_CONFIG['epochs']} epochs")
print(f"Batch      : {TRAINING_CONFIG['batch']} | Device: {TRAINING_CONFIG['device']}")
print(f"\nClasses ({NUM_CLASSES}):")
for i in range(NUM_CLASSES):
    print(f"  {i:2d}: {CATEGORIES[i + 1]}")
print(f"\nMLflow")
print(f"  Experiment : {MLFLOW_EXPERIMENT}")
print(f"  Tracking   : {MLFLOW_TRACKING_URI}")
print(f"  Dashboard  : mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print(f"  Then open  : http://localhost:5000")
